# ✏️ Draw & Guess — Sketch Classifier
A CNN trained on Google's QuickDraw dataset that recognises hand-drawn doodles across 15 categories.

**What's new vs previous projects:**
- Multi-class classification (15 categories, not binary)
- Building a custom dataset by downloading and sampling `.npy` files
- Sparse categorical cross-entropy loss
- A Gradio app where you draw and the model guesses in real time

**Run everything:** Runtime → Run all (`Ctrl+F9`) — enable GPU first!

| Step | What happens |
|------|---------------|
| 1 | Install & import |
| 2 | Download QuickDraw dataset |
| 3 | Explore & visualize |
| 4 | Preprocess & build tf.data pipeline |
| 5 | Build the CNN |
| 6 | Train with early stopping |
| 7 | Evaluate — curves, confusion matrix, mistakes |
| 8 | Save the model |
| 9 | Draw & guess live (Gradio) |

## Step 1 — Install & Import Libraries

In [ ]:
!pip install tensorflow gradio seaborn pillow --quiet

import os
import random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from PIL import Image

from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import (
    Conv2D, MaxPooling2D, Flatten,
    Dense, Dropout, BatchNormalization,
    RandomRotation, RandomZoom, RandomTranslation
)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

print('✅ All libraries imported!')
print(f'   TensorFlow: {tf.__version__}')

## Step 2 — Download the QuickDraw Dataset

In [ ]:
# QuickDraw data is stored as .npy files on Google Cloud Storage
# Each file is a (N, 784) array — N drawings, each flattened to 28x28 = 784 pixels
# We download 15 categories and sample 5,000 drawings from each

CATEGORIES = [
    'cat', 'dog', 'car', 'house', 'tree',
    'sun', 'fish', 'bird', 'bicycle', 'airplane',
    'pizza', 'umbrella', 'clock', 'star', 'apple'
]

SAMPLES_PER_CLASS = 5000
BASE_URL = 'https://storage.googleapis.com/quickdraw_dataset/full/numpy_bitmap/'

X_list, y_list = [], []

for idx, category in enumerate(CATEGORIES):
    url  = BASE_URL + category.replace(' ', '%20') + '.npy'
    path = f'{category}.npy'

    if not os.path.exists(path):
        os.system(f'wget -q "{url}" -O "{path}"')

    data = np.load(path)                       # shape: (N, 784)
    data = data[:SAMPLES_PER_CLASS]            # take first 5000
    X_list.append(data)
    y_list.append(np.full(len(data), idx))     # label = index in CATEGORIES

    print(f'  [{idx+1:2d}/15] {category:<12} — {len(data):,} drawings loaded')

X = np.concatenate(X_list, axis=0)            # shape: (75000, 784)
y = np.concatenate(y_list, axis=0)            # shape: (75000,)

print(f'\n✅ Dataset ready!')
print(f'   Total drawings : {len(X):,}')
print(f'   Classes        : {len(CATEGORIES)}')

## Step 3 — Explore & Visualize

In [ ]:
# Show 3 samples from each of the 15 categories
fig, axes = plt.subplots(15, 3, figsize=(6, 40))
for row, category in enumerate(CATEGORIES):
    class_indices = np.where(y == row)[0]
    samples = np.random.choice(class_indices, 3, replace=False)
    for col, idx in enumerate(samples):
        axes[row, col].imshow(X[idx].reshape(28, 28), cmap='gray_r')
        axes[row, col].axis('off')
        if col == 0:
            axes[row, col].set_title(category, fontsize=9, loc='left')
plt.suptitle('3 samples per category', y=1.005, fontsize=13)
plt.tight_layout()
plt.show()

# Class distribution
plt.figure(figsize=(12, 3))
counts = [np.sum(y == i) for i in range(len(CATEGORIES))]
plt.bar(CATEGORIES, counts, color='steelblue', edgecolor='white')
plt.title('Samples per category')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print('💡 Dataset is perfectly balanced — 5,000 samples per class.')

## Step 4 — Preprocess & Build tf.data Pipeline

In [ ]:
# Reshape: (75000, 784) → (75000, 28, 28, 1)
X = X.reshape(-1, 28, 28, 1).astype('float32')

# QuickDraw stores drawings as white strokes on black background
# The raw pixel values: 255 = stroke, 0 = background
# Normalize to 0.0–1.0
X = X / 255.0

# Train / validation / test split: 70% / 15% / 15%
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=SEED, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=SEED, stratify=y_temp
)

# Build tf.data datasets for fast GPU feeding
BATCH_SIZE   = 128
AUTOTUNE     = tf.data.AUTOTUNE

data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomRotation(0.1, seed=SEED),
    tf.keras.layers.RandomZoom(0.1, seed=SEED),
    tf.keras.layers.RandomTranslation(0.1, 0.1, seed=SEED),
])

train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train))
train_ds = train_ds.shuffle(10000, seed=SEED)
train_ds = train_ds.batch(BATCH_SIZE)
train_ds = train_ds.map(
    lambda x, y: (data_augmentation(x, training=True), y),
    num_parallel_calls=AUTOTUNE
).prefetch(AUTOTUNE)

val_ds = tf.data.Dataset.from_tensor_slices((X_val, y_val))
val_ds = val_ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

test_ds = tf.data.Dataset.from_tensor_slices((X_test, y_test))
test_ds = test_ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

print('✅ Preprocessing complete!')
print(f'   Train   : {len(X_train):,} samples')
print(f'   Val     : {len(X_val):,} samples')
print(f'   Test    : {len(X_test):,} samples')
print(f'   Shape   : {X_train.shape[1:]}')

## Step 5 — Build the CNN Model

In [ ]:
# Architecture similar to the digit recognizer but with 15 output classes
# Key difference: Dense(15, softmax) + sparse_categorical_crossentropy loss

model = Sequential([
    # --- Block 1: detect strokes and edges ---
    Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=(28, 28, 1)),
    BatchNormalization(),
    Conv2D(32, (3, 3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D((2, 2)),
    Dropout(0.25),

    # --- Block 2: detect shapes and structure ---
    Conv2D(64, (3, 3), activation='relu', padding='same'),
    BatchNormalization(),
    Conv2D(64, (3, 3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D((2, 2)),
    Dropout(0.25),

    # --- Block 3: detect full object structure ---
    Conv2D(128, (3, 3), activation='relu', padding='same'),
    BatchNormalization(),
    Dropout(0.25),

    # --- Classification head ---
    Flatten(),
    Dense(256, activation='relu'),
    BatchNormalization(),
    Dropout(0.5),
    Dense(15, activation='softmax')  # 15 output neurons, one per category
])

# sparse_categorical_crossentropy: labels are integers (not one-hot)
# This is more memory-efficient than to_categorical for many classes
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.build(input_shape=(None, 28, 28, 1))
model.summary()

## Step 6 — Train the Model

In [ ]:
early_stop = EarlyStopping(
    monitor='val_loss', patience=5,
    restore_best_weights=True, verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss', factor=0.5,
    patience=2, min_lr=1e-6, verbose=1
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=30,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

print(f'\n✅ Training complete — stopped at epoch {len(history.history["loss"])}')

## Step 7 — Evaluate the Model

In [ ]:
# Test accuracy
loss, acc = model.evaluate(test_ds, verbose=0)
print(f'Test Accuracy : {acc*100:.2f}%')
print(f'Test Loss     : {loss:.4f}')
print(f'Chance level  : {100/len(CATEGORIES):.1f}% (random guessing)\n')

# Training curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history['accuracy'],     label='Train')
axes[0].plot(history.history['val_accuracy'], label='Validation')
axes[0].set_title('Accuracy over epochs')
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[1].plot(history.history['loss'],     label='Train')
axes[1].plot(history.history['val_loss'], label='Validation')
axes[1].set_title('Loss over epochs')
axes[1].set_xlabel('Epoch')
axes[1].legend()
plt.tight_layout()
plt.show()

# Predictions on test set
y_pred = np.argmax(model.predict(test_ds, verbose=0), axis=1)
print(classification_report(y_test, y_pred, target_names=CATEGORIES))

# Confusion matrix (15x15)
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CATEGORIES, yticklabels=CATEGORIES)
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

## Step 7b — Visualize Mistakes

In [ ]:
# Find wrong predictions
wrong_idx = np.where(y_pred != y_test)[0]
print(f'Total mistakes: {len(wrong_idx)} / {len(y_test)} ({len(wrong_idx)/len(y_test)*100:.2f}%)')

plt.figure(figsize=(14, 8))
for i, idx in enumerate(wrong_idx[:20]):
    plt.subplot(4, 5, i + 1)
    plt.imshow(X_test[idx].reshape(28, 28), cmap='gray_r')
    true_label = CATEGORIES[y_test[idx]]
    pred_label = CATEGORIES[y_pred[idx]]
    plt.title(f'True: {true_label}\nPred: {pred_label}', fontsize=7, color='crimson')
    plt.axis('off')
plt.suptitle('Sketches the model got wrong', fontsize=13)
plt.tight_layout()
plt.show()

## Step 7c — Learning Rate History

In [ ]:
lr_history = history.history.get('learning_rate', history.history.get('lr', []))

if lr_history:
    plt.figure(figsize=(8, 3))
    plt.plot(lr_history, marker='o', color='darkorange')
    plt.title('Learning rate over epochs')
    plt.xlabel('Epoch')
    plt.ylabel('Learning rate')
    plt.yscale('log')
    plt.tight_layout()
    plt.show()

    drops = [i for i in range(1, len(lr_history)) if lr_history[i] < lr_history[i-1]]
    if drops:
        print(f'LR reduced at epoch(s): {[d+1 for d in drops]}')
    else:
        print('LR was never reduced — training converged smoothly!')

## Step 8 — Save the Model

In [ ]:
model.save('sketch_classifier.keras')
print('✅ Model saved as sketch_classifier.keras')

# To reload:
# model = load_model('sketch_classifier.keras')

## Step 9 — Draw & Guess Live (Gradio)
> Draw any of the 15 categories in the sketchpad. The model shows its top 3 guesses with confidence bars.

In [ ]:
import gradio as gr

def predict_sketch(img):
    if img is None:
        return {c: 0.0 for c in CATEGORIES}

    # Handle both dict (newer Gradio) and array inputs
    if isinstance(img, dict):
        img = img.get('composite', img.get('background', None))
    if img is None:
        return {c: 0.0 for c in CATEGORIES}

    # Resize to 28x28 grayscale using PIL
    pil_img = Image.fromarray(img.astype('uint8'))
    pil_img = pil_img.convert('L')
    pil_img = pil_img.resize((28, 28), Image.LANCZOS)

    arr = np.array(pil_img).astype('float32')

    # QuickDraw trains on white strokes on black background
    # Sketchpad draws black strokes on white background — invert!
    arr = 255.0 - arr
    arr = arr / 255.0
    arr = arr.reshape(1, 28, 28, 1)

    probs = model.predict(arr, verbose=0)[0]
    return {CATEGORIES[i]: float(probs[i]) for i in range(len(CATEGORIES))}


with gr.Blocks() as app:
    gr.Markdown(
        '# ✏️ Draw & Guess\n'
        'Draw one of these: **cat · dog · car · house · tree · sun · fish · bird · bicycle · airplane · pizza · umbrella · clock · star · apple**'
    )
    with gr.Row():
        with gr.Column():
            sketchpad = gr.Sketchpad(
                label='Draw here',
                type='numpy'
            )
            clear_btn = gr.Button('Clear')
        with gr.Column():
            output = gr.Label(
                num_top_classes=3,
                label='Top 3 guesses'
            )

    sketchpad.change(fn=predict_sketch, inputs=sketchpad, outputs=output)
    clear_btn.click(fn=lambda: None, inputs=None, outputs=sketchpad)

app.launch(debug=False)